# Figure S1C — Pre-RAG vs Post-RAG F1 Comparison (110 Patients)

Grouped bar chart comparing F1 scores (with 95% stratified bootstrap CIs) across 6 toxicities for pre-RAG and post-RAG pipelines, with paired bootstrap p-values.

**Data sources** (under `figures/figures_data/figure 1/data`):
- `110_patients_results_no_rag.csv` — pre-RAG model predictions
- `110_patients_results.csv` — post-RAG model predictions
- `llm_train_dates_subset.csv` — gold standard patient labels

Outputs are written to `figure 1/results/supp/`.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, fbeta_score, f1_score

import matplotlib

%matplotlib inline

matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42

import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")


In [ ]:
# ---------------------------------------------------------------------------
# FILE PATHS — notebook lives in figure 1/scripts/
#   figures/
#   ├── figures_data/figure 1/data/   ← inputs (shared OneDrive data dir)
#   └── v1/figure 1/
#       ├── scripts/                  ← this notebook
#       └── results/supp/             ← output PDF + CSV
# ---------------------------------------------------------------------------
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA = FIGURES / "figures_data" / "figure 1" / "data"
RESULTS = ROOT / "results"
(RESULTS / "supp").mkdir(parents=True, exist_ok=True)

MODEL_PRE  = DATA / "110_patients_results_no_rag.csv"
MODEL_POST = DATA / "110_patients_results.csv"
GOLD_CSV   = DATA / "llm_train_dates_subset.csv"
OUT_PATH   = RESULTS / "supp" / "Pre_vs_Post_RAG_S1C.pdf"

for p in [MODEL_PRE, MODEL_POST, GOLD_CSV]:
    assert p.exists(), f"Missing: {p}"
print("All input files found.")
print(f"Data: {DATA}")
print(f"Results: {RESULTS / 'supp'}")


In [ ]:
# ---------------------------------------------------------------------------
# CONSTANTS
# ---------------------------------------------------------------------------
N_BOOT = 2000
SEED   = 42

TOXICITIES = [
    "liver_toxicity",
    "hypothyroidism",
    "pneumonitis",
    "colitis",
    "adrenal_insufficiency",
    "hyperthyroidism",
]

DISPLAY = {
    "pneumonitis":           "Pneumonitis",
    "adrenal_insufficiency": "Adrenal\ninsufficiency",
    "liver_toxicity":        "Liver\ntoxicity",
    "colitis":               "Colitis",
    "hyperthyroidism":       "Hyper-\nthyroidism",
    "hypothyroidism":        "Hypo-\nthyroidism",
}

PRE_COLOR  = "#E8955A"
POST_COLOR = "#4878CF"


In [ ]:
def _read_csv(path, usecols=None):
    kwargs = {"low_memory": False}
    if usecols is not None:
        kwargs["usecols"] = usecols
    try:
        return pd.read_csv(path, **kwargs)
    except UnicodeDecodeError:
        for enc in ("latin-1", "cp1252"):
            try:
                return pd.read_csv(path, encoding=enc, **kwargs)
            except UnicodeDecodeError:
                continue
        raise

def norm_mrn(s):
    return s.astype(str).str.replace(r"\.0$", "", regex=True).str.strip().str.zfill(8)


In [ ]:
def find_optimal_threshold(y_true, y_score):
    thresholds = np.linspace(0, 1, 101)
    best_score, best_thr = 0.0, 0.5
    for thr in thresholds:
        y_pred = (y_score >= (thr - 1e-10)).astype(int)
        score = fbeta_score(y_true, y_pred, beta=1, zero_division=0)
        if score > best_score:
            best_score = score
            best_thr = float(thr)
    return best_thr, best_score

def f1_at_threshold(y_true, y_score, thr):
    y_pred = (y_score >= (thr - 1e-10)).astype(int)
    return float(f1_score(y_true, y_pred, zero_division=0))

def calculate_metrics_at_threshold(y_true, y_score, threshold):
    y_pred = (y_score >= (threshold - 1e-10)).astype(int)
    if len(np.unique(y_true)) < 2:
        if np.all(y_true == 0):
            tn, fp = int(np.sum(y_pred == 0)), int(np.sum(y_pred == 1))
            tp = fn = 0
        else:
            tp, fn = int(np.sum(y_pred == 1)), int(np.sum(y_pred == 0))
            tn = fp = 0
    else:
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"TP": tp, "FP": fp, "TN": tn, "FN": fn,
            "Precision": precision, "Recall": recall, "F1_Score": f1, "Threshold": threshold}


In [ ]:
def stratified_bootstrap_with_samples(y_true, y_score, thr, n_boot, rng):
    """Return (point_f1, ci_low, ci_high, boot_f1_array)."""
    point = f1_at_threshold(y_true, y_score, thr)
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]

    # Store resampled indices for paired testing
    boot_f1 = np.empty(n_boot)
    boot_indices = []
    for b in range(n_boot):
        s_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True) if len(pos_idx) > 0 else pos_idx
        s_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([s_pos, s_neg])
        boot_indices.append(idx)
        boot_f1[b] = f1_at_threshold(y_true[idx], y_score[idx], thr)

    lo = float(np.percentile(boot_f1, 2.5))
    hi = float(np.percentile(boot_f1, 97.5))
    return point, lo, hi, boot_f1, boot_indices


In [ ]:
def load_scores_and_truth(model_csv, gold_csv):
    model_cols = lambda c: c.lower() == "mrn" or c in (
        "liver toxicity", "adrenal insufficiency",
        "colitis", "hyperthyroidism", "hypothyroidism", "pneumonitis",
    )
    df_model = _read_csv(model_csv, usecols=model_cols)
    df_model = df_model.rename(columns={
        "liver toxicity": "liver_toxicity",
        "adrenal insufficiency": "adrenal_insufficiency",
    })
    df_model["mrn"] = norm_mrn(df_model["mrn"])

    gold_cols = lambda c: c.lower() in ("mrn", "mrn_str") or c in TOXICITIES
    df_gold_raw = _read_csv(gold_csv, usecols=gold_cols)
    mrn_col = next(c for c in ("MRN", "mrn", "MRN_STR") if c in df_gold_raw.columns)
    df_gold_raw["mrn"] = norm_mrn(df_gold_raw[mrn_col])

    for tox in TOXICITIES:
        df_gold_raw[tox] = pd.to_numeric(df_gold_raw[tox], errors="coerce").fillna(0).astype(int)

    patient_gold = df_gold_raw.groupby("mrn")[TOXICITIES].max()
    shared = sorted(set(df_model["mrn"]) & set(patient_gold.index))
    if not shared:
        raise ValueError("No patients overlap between model and gold")

    scores = df_model[df_model["mrn"].isin(shared)].groupby("mrn")[TOXICITIES].max().sort_index()
    truth = patient_gold.loc[shared].astype(int)
    return scores, truth

print("Loading pre-RAG...")
scores_pre, truth_pre = load_scores_and_truth(MODEL_PRE, GOLD_CSV)
print(f"  {len(scores_pre)} patients")

print("Loading post-RAG...")
scores_post, truth_post = load_scores_and_truth(MODEL_POST, GOLD_CSV)
print(f"  {len(scores_post)} patients")


In [ ]:
# ---------------------------------------------------------------------------
# Compute metrics + paired bootstrap p-values
# ---------------------------------------------------------------------------
# Use a shared RNG seed so both pre and post get the SAME resampled indices
# per bootstrap iteration — this is what makes it a paired test.

results_pre = []
results_post = []
p_values = []

for tox in TOXICITIES:
    y_true = truth_pre[tox].values.astype(int)
    s_pre  = scores_pre[tox].values.astype(float)
    s_post = scores_post[tox].values.astype(float)

    # Find optimal thresholds
    if y_true.sum() == 0:
        thr_pre = thr_post = 0.5
    else:
        thr_pre, _  = find_optimal_threshold(y_true, s_pre)
        thr_post, _ = find_optimal_threshold(y_true, s_post)

    m_pre  = calculate_metrics_at_threshold(y_true, s_pre, thr_pre)
    m_post = calculate_metrics_at_threshold(y_true, s_post, thr_post)

    # Paired bootstrap: same indices for both
    rng = np.random.default_rng(SEED)
    pos_idx = np.where(y_true == 1)[0]
    neg_idx = np.where(y_true == 0)[0]

    boot_pre  = np.empty(N_BOOT)
    boot_post = np.empty(N_BOOT)
    for b in range(N_BOOT):
        s_pos = rng.choice(pos_idx, size=len(pos_idx), replace=True) if len(pos_idx) > 0 else pos_idx
        s_neg = rng.choice(neg_idx, size=len(neg_idx), replace=True)
        idx = np.concatenate([s_pos, s_neg])
        boot_pre[b]  = f1_at_threshold(y_true[idx], s_pre[idx], thr_pre)
        boot_post[b] = f1_at_threshold(y_true[idx], s_post[idx], thr_post)

    # CIs
    pre_lo, pre_hi   = np.percentile(boot_pre, 2.5), np.percentile(boot_pre, 97.5)
    post_lo, post_hi = np.percentile(boot_post, 2.5), np.percentile(boot_post, 97.5)

    # Two-sided paired bootstrap p-value
    delta = boot_post - boot_pre
    p_val = 2 * min(np.mean(delta >= 0), np.mean(delta <= 0))
    p_val = min(p_val, 1.0)  # cap at 1

    results_pre.append({"tox": tox, "F1": m_pre["F1_Score"], "lo": pre_lo, "hi": pre_hi})
    results_post.append({"tox": tox, "F1": m_post["F1_Score"], "lo": post_lo, "hi": post_hi})
    p_values.append(p_val)

    print(f"  {tox:<25} pre F1={m_pre['F1_Score']:.3f} [{pre_lo:.3f}-{pre_hi:.3f}]  "
          f"post F1={m_post['F1_Score']:.3f} [{post_lo:.3f}-{post_hi:.3f}]  "
          f"p={p_val:.4f}")


In [ ]:
def format_pval(p):
    """Format p-value for Nature-style display."""
    if p < 0.001:
        return "P < 0.001"
    elif p < 0.01:
        return f"P = {p:.3f}"
    elif p < 0.05:
        return f"P = {p:.2f}"
    else:
        return f"P = {p:.2f}"

for tox, p in zip(TOXICITIES, p_values):
    print(f"  {tox}: {format_pval(p)}")


In [ ]:
# ---------------------------------------------------------------------------
# Figure with p-value brackets
# ---------------------------------------------------------------------------
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "legend.fontsize": 5,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

fig, ax = plt.subplots(figsize=(3.6, 2.0))  
x = np.arange(len(TOXICITIES))
w = 0.36

pre_f1  = np.array([r["F1"] for r in results_pre])
pre_lo  = np.array([r["lo"] for r in results_pre])
pre_hi  = np.array([r["hi"] for r in results_pre])
pre_yerr = np.array([np.maximum(0.0, pre_f1 - pre_lo), np.maximum(0.0, pre_hi - pre_f1)])

post_f1  = np.array([r["F1"] for r in results_post])
post_lo  = np.array([r["lo"] for r in results_post])
post_hi  = np.array([r["hi"] for r in results_post])
post_yerr = np.array([np.maximum(0.0, post_f1 - post_lo), np.maximum(0.0, post_hi - post_f1)])

bars_pre = ax.bar(x - w/2, pre_f1, width=w, yerr=pre_yerr, capsize=3,
       color=PRE_COLOR, edgecolor="white", linewidth=0.5,
       error_kw={"linewidth": 0.8}, label="Pre-RAG")
bars_post = ax.bar(x + w/2, post_f1, width=w, yerr=post_yerr, capsize=3,
       color=POST_COLOR, edgecolor="white", linewidth=0.5,
       error_kw={"linewidth": 0.8}, label="Post-RAG")

# --- P-value brackets ---
bracket_color = "black"
bracket_lw = 0.8

for i in range(len(TOXICITIES)):
    # Top of each bar + error bar
    top_pre  = pre_f1[i] + pre_yerr[1, i]
    top_post = post_f1[i] + post_yerr[1, i]
    bracket_y = max(top_pre, top_post) + 0.03  # bracket base
    bracket_top = bracket_y + 0.02              # bracket height

    x_left  = x[i] - w/2
    x_right = x[i] + w/2

    # Horizontal bracket with vertical ticks
    ax.plot([x_left, x_left, x_right, x_right],
            [bracket_y, bracket_top, bracket_top, bracket_y],
            color=bracket_color, linewidth=bracket_lw, clip_on=False)

    # P-value text
    ax.text((x_left + x_right) / 2, bracket_top + 0.01,
            format_pval(p_values[i]),
            ha="center", va="bottom", fontsize=5, fontfamily="Arial",
            fontstyle="italic")

ax.set_ylabel("F1 Score")
ax.set_xticks(x)
ax.set_xticklabels([DISPLAY[t] for t in TOXICITIES])
ax.set_ylim(0, 1.25)  # extra room for brackets
ax.set_yticks(np.arange(0, 1.1, 0.2))
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)
for spine in ("bottom", "left"):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)
ax.legend(frameon=False, loc="upper right")

plt.tight_layout()
fig.savefig(OUT_PATH, format="pdf", dpi=450)
print(f"Saved: {OUT_PATH}")
plt.show()

In [ ]:
CSV_OUT = OUT_PATH.with_name(OUT_PATH.stem + "_results.csv")

export_df = pd.DataFrame({
    "toxicity": TOXICITIES,
    "toxicity_display": [DISPLAY[t].replace("\n", " ") for t in TOXICITIES],
    "pre_rag_f1":      [r["F1"] for r in results_pre],
    "pre_rag_ci_low":  [r["lo"] for r in results_pre],
    "pre_rag_ci_high": [r["hi"] for r in results_pre],
    "post_rag_f1":      [r["F1"] for r in results_post],
    "post_rag_ci_low":  [r["lo"] for r in results_post],
    "post_rag_ci_high": [r["hi"] for r in results_post],
    "p_value_paired_bootstrap": p_values,
})

export_df["delta_f1"] = export_df["post_rag_f1"] - export_df["pre_rag_f1"]
export_df["n_patients"] = len(truth_pre)
export_df["n_positive"] = [int(truth_pre[t].sum()) for t in TOXICITIES]
export_df["n_bootstrap"] = N_BOOT
export_df["seed"] = SEED

export_df.to_csv(CSV_OUT, index=False)
print(f"Saved: {CSV_OUT.name}")
print(f"  {len(export_df)} toxicities, {export_df['n_patients'].iloc[0]} patients")